## Train and test the Threat Detekt detector

This notebook trains a YOLOv8m detector, then scores a saved checkpoint on the test split.

Run it from the `model-training` directory after `config.yaml` is filled in. `path` is the dataset root. `train`, `val`, and `test` are image folders relative to that root. The class names in that file are person, handgun, rifle, shotgun, and knife. The training settings are in the training cell below. `config.yaml` does not set them.

The notebook needs the `ultralytics` and `torch` packages. The training cell is set up for a CUDA GPU.

In [ ]:
import torch
from ultralytics import YOLO

# Report whether this kernel can see a CUDA GPU. Training still picks the device itself.
if torch.cuda.is_available():
    print("CUDA is available! PyTorch can use the GPU.")
else:
    print("CUDA is not available. PyTorch will use the CPU.")

In [ ]:
torch.cuda.empty_cache()  # drop unused cached GPU memory; tensors still in use stay allocated

### Train

The next cell builds YOLOv8m from `yolov8m.yaml`. That file is the architecture only, so this run starts without the COCO checkpoint `yolov8m.pt`.

Training reads `config.yaml` from the working directory and writes the run to `runs/detect/train13/`. A later run with the same name reuses that folder. It trains for up to 150 epochs and stops earlier when the validation split shows no improvement for 20 epochs.

Images are resized to 640 pixels. Each step uses 4 images, and gradients accumulate until the effective batch size is 64. The full training split is used. Images are cached in memory, with 8 loader workers, and training runs in mixed precision.

The best and last checkpoints are saved, and another checkpoint is written every 25 epochs. Plots are saved in the same folder.

Images may be rotated by up to 10 degrees. Mosaic stays on until the final 10 epochs. A horizontal flip is applied half the time. Mixup runs on about 10% of samples and copy-paste on about 30%. Vertical flip and shear are off. A small perspective warp is applied. The learning rate starts at 0.01 and finishes at one tenth of that. The first 3 epochs are warmup.

The line printed at the end is mAP50 for `best.pt` on the `val` split. The test split is scored in the next section. Change the values in the cell to adjust the run.

In [ ]:
model = YOLO("yolov8m.yaml")  # architecture only; does not load COCO weights from yolov8m.pt
results = model.train(
        data="config.yaml",
        epochs=150,
        patience=20,  # stop when the val split shows no improvement for 20 epochs
        batch=4,
        nbs=64,  # accumulate gradients to an effective batch size of 64
        imgsz=640,
        workers=8,
        cache=True,  # cache images in RAM
        project="runs/detect",
        name="train13",
        exist_ok=True,  # reuse runs/detect/train13 on a later run
        plots=True,
        save=True,  # write best.pt and last.pt
        save_period=25,  # also write a checkpoint every 25 epochs
        lr0=0.01,
        lrf=0.1,  # final learning rate is lr0 * lrf
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        warmup_momentum=0.8,
        warmup_bias_lr=0.1,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        degrees=10.0,  # rotate up to 10 degrees for camera angles
        translate=0.1,
        scale=0.5,
        shear=0.0,
        perspective=0.0005,  # slight perspective warp
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        copy_paste=0.3,
        box=7.5,
        cls=0.7,
        dfl=1.5,
        close_mosaic=10,  # turn mosaic off for the last 10 epochs
        amp=True,  # mixed precision
        fraction=1.0,  # use the full training split
    )

# mAP50 of best.pt on the val split. The test split is scored in the next section.
print(f"Training completed. Best mAP50: {results.box.map50:.4f}")

### Test

The next cell loads one checkpoint and runs it on the `test` split in `config.yaml`. A detection is kept when its confidence is at least 0.25. Overlapping boxes are suppressed at an IoU of 0.6, and each image keeps at most 300 boxes. The image size is 640, the same size used in training.

The checkpoint path in the cell is `runs/detect/train1/weights/best.pt`, joined to the directory of `__file__`. The training cell writes `runs/detect/train13/weights/best.pt` in the working directory. Set `model_path` to the checkpoint you want to score. This notebook does not define `__file__`, so use a normal path such as `runs/detect/train13/weights/best.pt`.

The cell prints mAP50, mAP50-95, precision, and recall for the whole test split, then one line per class name in `model.names` order. A class line is printed when that index exists in the per-class score list.

Plots and `predictions.json` are written under `runs/detect/`. The requested folder name is `test_validation`. When that folder already exists, Ultralytics adds a number to the name. The last printed line still says `runs/detect/test_validation/`. `save_hybrid=True` is passed and then dropped; current Ultralytics does not write hybrid labels for that argument.

In [ ]:
import os

import torch
from ultralytics import YOLO

torch.cuda.empty_cache()  # drop unused cached GPU memory before validation

# Looks for runs/detect/train1/weights/best.pt next to __file__.
# The training cell writes runs/detect/train13/weights/best.pt in the working directory.
# Jupyter does not define __file__; set model_path to the checkpoint you want to score.
model_path = os.path.join(
    os.path.dirname(__file__), "runs", "detect", "train1", "weights", "best.pt"
)
if not os.path.exists(model_path):
    raise ValueError(f"Model file does not exist: {model_path}")

print(f"Loading model from: {model_path}")
model = YOLO(model_path)

# Run validation on test data
print("Running validation on test data...")
results = model.val(
    data="config.yaml",
    split="test",  # images listed under test: in config.yaml
    imgsz=640,  # same image size as the training cell
    conf=0.25,
    iou=0.6,
    max_det=300,
    project="runs/detect",
    name="test_validation",
    save_json=True,  # write predictions.json in the results folder
    save_hybrid=True,  # ignored; current Ultralytics removes this argument
    plots=True,  # save validation plots
    verbose=True,  # print Ultralytics' own per-class table
)

# Print detailed results
print("\n" + "=" * 50)
print("VALIDATION RESULTS ON TEST DATA")
print("=" * 50)

# Print overall metrics
print(f"mAP50: {results.box.map50:.4f}")
print(f"mAP50-95: {results.box.map:.4f}")
print(f"Precision: {results.box.mp:.4f}")
print(f"Recall: {results.box.mr:.4f}")

# Class names follow model.names order. The score at that same index is printed
# when the index exists in the per-class arrays.
print("\nPer-class metrics:")
for i, class_name in enumerate(model.names.values()):
    if i < len(results.box.ap50):
        print(
            f"{class_name}: mAP50={results.box.ap50[i]:.4f}, mAP50-95={results.box.ap[i]:.4f}"
        )

# Print confusion matrix info
if hasattr(results, "confusion_matrix"):
    if hasattr(results.confusion_matrix, "matrix"):
        print(f"\nConfusion Matrix Shape: {results.confusion_matrix.matrix.shape}")
    else:
        print("\nConfusion Matrix object does not have a 'matrix' attribute.")

# This text stays the same when Ultralytics increments the folder name.
print("\nResults saved to: runs/detect/test_validation/")